## Defensa de la Evaluación

### 1. Introducción a la defensa
En esta defensa quiero explicar brevemente las decisiones que tomé al diseñar y programar cada ejercicio de la práctica. Mi objetivo principal fue aplicar bien los pilares (herencia, poliformismo, encampsulación y composición) para lograr un código limpio, fácil de mantener y que se puedan extender en el futuro sin complicar la estructura. 

### 2.Explicación y Justificación por Ejercicios

#### Ejercicio 3: Figuras Geométricas y Reutilización
- **Poliformismo**: 
Diseñé las clases `Rectangulo`, `Circulo` y `TrianguloRectangulo` para que compartan los mismos métodos clave (`àrea()` y `perimetro()`). De esta forma, la función `imprimir-informe()` puede trabajar con cualquiera de ellas de manera transparente. 

- **Herencia en `Cuadrado`:** 
Para crear la clase `Cuadrado`, decidí hacerla heredar directamente de `Rectángulo`. Como un cuadrado no deja de ser un rectángulo especial con los lados iguales, al llamar al constructor del padre mediante `super().__init__(lado, lado)` aprovecho directamente el cálculo del área y del perímetro. Así evito duplicar código. 

#### Ejercicio 5: Descuentos en Pedidos:
- **Separación de responsabilidades:** 
En lugar de recargar la clase `Pedido` con la lógica de los descuentos, preferí extraer el cálculo a cada tipo de cliente (`ClienteNormal`, `ClienteVIP`, `ClienteEstudiante`). Cada cliente sabe cómo calcular su propio total.

- **Extensibilidad:**
Al delegar esta responsabilidad, cuando tuve que añadir `ClienteEstudiante` no hizo falta tocar para nada la clase `Pedido`.

- **Autocrítica sobre la entrega:**
Es verdad que en el archivo inicial dejé la primera versión de `Pedido` antes de hacer el refactor. En un código final de producción lo ideal es borrar esa versión previa para evitar confusiones o redefiniciones innecesarias.


#### Ejercicio 7: Gestión de Cursos:
- **Estructura de clases:**
Creé las clases `Alumno` y `Profesor` heredando de una clase base común, `Persona`.

- **Validaciones de negocio:**
En la clase `Curso` utilicé comprobaciones de tipo (`isinstance`) para asegurarme de que solo se puedan matricular alumnos y que el docente sea un profesor. También añadí control para no matricular al mismo alumno dos veces y no superar la capacidad máxima.

- **Mejora:**
Para pulir el código, se podría añadir una validación en el constructor que obligue a que la capacidad máxima sea un número positivo (`capacidad_maxima > 0`), lanzando un error (`ValueError`) si no se cumple.

#### Ejercicio 8: MRO
- **Aclaración sobre mi código:**
En mi entrega cometí el detalle de instanciar `obj = D()` antes de modificar la línea donde redefinía la clase a `D(C, B)`. En Python, los objetos guardan la referencia a la clase con la que fueron creados. Por eso, aunque refefinamos la clase después, el cobjeto `obj` original sigue apuntando a `D(B, C)` y sigue devolviendo `DBCA`. 

**Cómo se soluciona correctamente:**
Para ver el cambio a `DCBA`, basta con crear un **nuevo objeto** después de refefinir la clase, o modifica rla definición y reiniciar la ejecución del script. 

In [1]:
class A:
    def metodo(self):
        return "A"

class B(A):
    def metodo(self):
        return "B" + super().metodo()

class C(A):
    def metodo(self):
        return "C" + super().metodo()

# 1. Primera versión: D hereda de (B, C)
class D(B, C):
    def metodo(self):
        return "D" + super().metodo()

obj1 = D()
print(obj1.metodo())  # Imprime: DBCA

# 2. Cambiamos el orden de herencia: D hereda de (C, B)
class D(C, B):
    def metodo(self):
        return "D" + super().metodo()

# Instanciamos un NUEVO objeto para usar el nuevo MRO
obj2 = D()
print(obj2.metodo())  # Imprime: DCBA

DBCA
DCBA


#### Ejercicio 10: Sistema de Productos y Pedidos: 
- **Uso del Polimorfismo:**
En la clase `Pedido` no utilizo comprobaciones de tipo(`if`, `type()`, etc.). Confío en que cualuquir producto (`ProductoFisico`, `ProductoDigital`, `Suscripción` o `ProductoDescuento`) implementa el método `calcular_precio()`.

- **Composición / Decorador:**
La clase `ProductoDescuento` envuelve a otro producto y calcula el descuento de la forma dinámica basándose en el precio que le devuelve el objeto original. 

- **Refactorización e Identificadores:**
Para no tener que asignar manualmente el identificador por fuera (`p_descuento.id = "D02"`), lo correcto es pasárselo directamente al constructor o generarlo automáticamente a partir del ID del producto base. 
Además, para proteger la lista de productos de modificaciones externas indeseadas, se encapsula como un atributo privado (`_productos`) y se ofrece acceso de solo lectura mediante una propiedad. 

In [2]:
class ProductoDescuento:
    def __init__(self, producto, porcentaje_descuento, id_descuento=None):
        self.producto = producto
        self.porcentaje_descuento = porcentaje_descuento
        # Asignamos el ID de forma limpia en la inicialización
        self.id = id_descuento if id_descuento else f"DESC-{producto.id}"

    def calcular_precio(self):
        precio_base = self.producto.calcular_precio()
        return precio_base * (1 - self.porcentaje_descuento / 100)

class Pedido:
    def __init__(self):
        self._productos = []  # Atributo protegido

    @property
    def productos(self):
        # Devolvemos una copia para proteger la lista interna
        return list(self._productos)

    def agregar_producto(self, producto):
        self._productos.append(producto)

    def calcular_total(self):
        # Polimorfismo: cada producto sabe cómo calcular su precio
        return sum(prod.calcular_precio() for prod in self._productos)

### 3. Conclusión
En resumen, con estas aclaraciones y ajustes busco demostrar que comprendo el funcionamiento de Python (como el MRO y el comportamiento de las instancias en memoria) y la importancia de aplicar un diseño enfocado en la encapsulación y el polimorfismo limpio. 